## Create elo model
Based on the final 22-23 elo posted by FiveThirtyEight.

Procedures:
1. Start from obtaining the final elo ratings after 22-23 season(includes championships)
2. Calculate elo game by game through the season 23-24, 24-25, 25-26
4. Implement the column of elo to main dataset

### 1. Reuse 23-26 seasons data, obtain 22-23 final elo manually from published data

In [16]:
# FiveThirtyEight Official Final Elo Ratings at the end of the 2022-23 Season (Post-Finals)
import pandas as pd

official_elo_2223 = {
    'BOS': 1709, 'DEN': 1686, 'PHX': 1640, 'PHI': 1638, 'MIA': 1632,
    'GSW': 1629, 'CLE': 1625, 'LAL': 1618, 'MEM': 1615, 'NYK': 1612,
    'LAC': 1604, 'SAC': 1585, 'MIN': 1558, 'NOP': 1552, 'OKC': 1538,
    'TOR': 1535, 'MIL': 1673, 'CHI': 1532, 'ATL': 1530, 'BKN': 1515,
    'DAL': 1510, 'UTA': 1465, 'WAS': 1438, 'IND': 1435, 'ORL': 1430,
    'POR': 1385, 'CHA': 1375, 'HOU': 1325, 'SAS': 1320, 'DET': 1285
}

# Convert to DataFrame to match your pipeline schema
final_elo_2223 = pd.DataFrame(
    list(official_elo_2223.items()),
    columns=["TEAM", "FINAL_ELO_2223"]
).sort_values("FINAL_ELO_2223", ascending=False).reset_index(drop=True)

In [17]:
single_game_reuse = pd.read_csv("single_game.csv")
df_2324 = single_game_reuse[single_game_reuse["SEASON"] == "2023-24"]
df_2425 = single_game_reuse[single_game_reuse["SEASON"] == "2024-25"]
df_2526 = single_game_reuse[single_game_reuse["SEASON"] == "2025-26"]

In [18]:
# only keep the columns we need
# modify elo_columns for 23-26 seasons
elo_columns1 = [
    "SEASON",
    "TEAM_ABBREVIATION_away",
    "TEAM_ABBREVIATION_home",
    "GAME_ID",
    "GAME_DATE",
    "PLUS_MINUS"
]

df_2324 = df_2324[elo_columns1].copy()
df_2425 = df_2425[elo_columns1].copy()
df_2526 = df_2526[elo_columns1].copy()

### 2. elo functions

Elo:
1. calculate expected_win_prob
2. margin_of_victory multiplier
3. using 1 and 2, update elo

In [19]:
## define the function to calculate expected_win_prob

def expected_win_prob(home_elo, away_elo):
    return 1 / (1 + 10 ** (-(home_elo + 100 - away_elo) / 400))

In [20]:
# define function fro MOV multiplier
import numpy as np

def mov_multiplier(point_diff, winner_elo, loser_elo):
    return (
        np.log(abs(point_diff) + 1)
        * 2.2 / (2.2 + 0.001 * (winner_elo - loser_elo))
    )

In [21]:
# define the elo update function
def calculate_elo_change(home_elo, away_elo, home_margin):
    H = 100
    K = 20

    home_prob = expected_win_prob(home_elo, away_elo)
    home_result = 1 if home_margin > 0 else 0

    if home_result == 1:
        winner_elo = home_elo + H
        loser_elo = away_elo
    else:
        winner_elo = away_elo
        loser_elo = home_elo + H

    point_diff = abs(home_margin)
    mov = mov_multiplier(point_diff, winner_elo, loser_elo)

    return K * mov * (home_result - home_prob)

### 3. Using the defined functions, calculate elo over the 23-24 season

#### a. 23-24 season

In [22]:
# Apply season-to-season carryover (75% retention + 25% regression to 1505)
elo_ratings_2324 = {
    team: 0.75 * rating + 0.25 * 1505
    for team, rating in official_elo_2223.items()
}

# run 2023-24 loop 
games_2324_elo = df_2324.sort_values(["GAME_DATE", "GAME_ID"]).reset_index(drop=True).copy()
games_2324_elo["HOME_ELO_PRE"] = None
games_2324_elo["AWAY_ELO_PRE"] = None
games_2324_elo["HOME_ELO_POST"] = None
games_2324_elo["AWAY_ELO_POST"] = None

for idx, row in games_2324_elo.iterrows():
    home = row["TEAM_ABBREVIATION_home"]
    away = row["TEAM_ABBREVIATION_away"]

    home_elo = elo_ratings_2324[home]
    away_elo = elo_ratings_2324[away]

    change = calculate_elo_change(
        home_elo, away_elo, row["PLUS_MINUS"]
    )

    games_2324_elo.loc[idx, "HOME_ELO_PRE"] = home_elo
    games_2324_elo.loc[idx, "AWAY_ELO_PRE"] = away_elo

    elo_ratings_2324[home] += change
    elo_ratings_2324[away] -= change

    games_2324_elo.loc[idx, "HOME_ELO_POST"] = elo_ratings_2324[home]
    games_2324_elo.loc[idx, "AWAY_ELO_POST"] = elo_ratings_2324[away]

# Finalize 2023-24 ratings
final_elo_2324 = pd.DataFrame(
    list(elo_ratings_2324.items()),
    columns=["TEAM", "FINAL_ELO_2324"]
).sort_values("FINAL_ELO_2324", ascending=False).reset_index(drop=True)

display(final_elo_2324)

,TEAM,FINAL_ELO_2324
0,BOS,1767.393422
1,DEN,1720.557001
2,PHX,1711.085181
3,GSW,1708.082018
4,OKC,1701.877432
5,DAL,1674.193590
6,NYK,1671.468461
7,MIN,1668.094217
8,LAL,1649.498464
9,IND,1643.351367


#### b. 24-25 season

In [23]:
# 2024-25 season 

# Initialize 2024-25 Elo ratings
elo_ratings_2425 = {
    team: 0.75 * rating + 0.25 * 1505
    for team, rating in elo_ratings_2324.items()
}

games_2425_elo = df_2425.sort_values(["GAME_DATE", "GAME_ID"]).reset_index(drop=True).copy()
games_2425_elo["HOME_ELO_PRE"] = None
games_2425_elo["AWAY_ELO_PRE"] = None
games_2425_elo["HOME_ELO_POST"] = None
games_2425_elo["AWAY_ELO_POST"] = None

# update each row
for idx, row in games_2425_elo.iterrows():
    home = row["TEAM_ABBREVIATION_home"]
    away = row["TEAM_ABBREVIATION_away"]

    home_elo = elo_ratings_2425[home]
    away_elo = elo_ratings_2425[away]

    change = calculate_elo_change(
        home_elo, away_elo, row["PLUS_MINUS"]
    )

    games_2425_elo.loc[idx, "HOME_ELO_PRE"] = home_elo
    games_2425_elo.loc[idx, "AWAY_ELO_PRE"] = away_elo

    elo_ratings_2425[home] += change
    elo_ratings_2425[away] -= change

    games_2425_elo.loc[idx, "HOME_ELO_POST"] = elo_ratings_2425[home]
    games_2425_elo.loc[idx, "AWAY_ELO_POST"] = elo_ratings_2425[away]

# finalize 2024-25 ratings
final_elo_2425 = pd.DataFrame(
    list(elo_ratings_2425.items()),
    columns=["TEAM", "FINAL_ELO_2425"]
).sort_values("FINAL_ELO_2425", ascending=False).reset_index(drop=True)

display(final_elo_2425)

,TEAM,FINAL_ELO_2425
0,OKC,1807.396943
1,LAC,1757.810049
2,BOS,1726.056493
3,MIN,1707.354832
4,CLE,1691.651954
5,IND,1657.159566
6,GSW,1656.852292
7,MIL,1653.154050
8,DEN,1643.270947
9,LAL,1635.923830


#### c. 25-26 season

In [24]:
# 2025-26 Season

# Initialize 2025-26 Elo ratings with 75% retention + 25% regression to 1505
elo_ratings_2526 = {
    team: 0.75 * rating + 0.25 * 1505
    for team, rating in elo_ratings_2425.items()
}

games_2526_elo = df_2526.sort_values(["GAME_DATE", "GAME_ID"]).reset_index(drop=True).copy()
games_2526_elo["HOME_ELO_PRE"] = None
games_2526_elo["AWAY_ELO_PRE"] = None
games_2526_elo["HOME_ELO_POST"] = None
games_2526_elo["AWAY_ELO_POST"] = None

# update each row
for idx, row in games_2526_elo.iterrows():
    home = row["TEAM_ABBREVIATION_home"]
    away = row["TEAM_ABBREVIATION_away"]

    home_elo = elo_ratings_2526[home]
    away_elo = elo_ratings_2526[away]

    change = calculate_elo_change(
        home_elo, away_elo, row["PLUS_MINUS"]
    )

    games_2526_elo.loc[idx, "HOME_ELO_PRE"] = home_elo
    games_2526_elo.loc[idx, "AWAY_ELO_PRE"] = away_elo

    elo_ratings_2526[home] += change
    elo_ratings_2526[away] -= change

    games_2526_elo.loc[idx, "HOME_ELO_POST"] = elo_ratings_2526[home]
    games_2526_elo.loc[idx, "AWAY_ELO_POST"] = elo_ratings_2526[away]

# Finalize 2025-26 ratings
final_elo_2526 = pd.DataFrame(
    list(elo_ratings_2526.items()),
    columns=["TEAM", "FINAL_ELO_2526"]
).sort_values("FINAL_ELO_2526", ascending=False).reset_index(drop=True)

display(final_elo_2526)

,TEAM,FINAL_ELO_2526
0,SAS,1817.320731
1,BOS,1762.065768
2,DET,1761.961419
3,DEN,1757.072242
4,OKC,1744.428542
5,CHA,1697.945708
6,NYK,1674.190316
7,LAL,1663.945368
8,CLE,1648.123939
9,HOU,1643.664760


### 4. Check for continuity

In [25]:
# define function to check for continuity over the elos for each season

def check_elo_continuity(games_elo):
    home = games_elo[
        ["GAME_ID", "GAME_DATE", "TEAM_ABBREVIATION_home",
         "HOME_ELO_PRE", "HOME_ELO_POST"]
    ].rename(columns={
        "TEAM_ABBREVIATION_home": "TEAM",
        "HOME_ELO_PRE": "ELO_PRE",
        "HOME_ELO_POST": "ELO_POST"
    })

    away = games_elo[
        ["GAME_ID", "GAME_DATE", "TEAM_ABBREVIATION_away",
         "AWAY_ELO_PRE", "AWAY_ELO_POST"]
    ].rename(columns={
        "TEAM_ABBREVIATION_away": "TEAM",
        "AWAY_ELO_PRE": "ELO_PRE",
        "AWAY_ELO_POST": "ELO_POST"
    })

    team_games = pd.concat([home, away], ignore_index=True)

    team_games = team_games.sort_values(
        ["TEAM", "GAME_DATE", "GAME_ID"]
    ).reset_index(drop=True)

    team_games["PREV_ELO_POST"] = (
        team_games.groupby("TEAM")["ELO_POST"].shift(1)
    )

    team_games["ELO_DIFF"] = (
        team_games["ELO_PRE"] - team_games["PREV_ELO_POST"]
    )

    # Exclude the first game for each team (no previous game to compare)
    checked = team_games.dropna(subset=["PREV_ELO_POST"])

    # Check whether all differences are approximately zero
    mismatches = checked[
        ~np.isclose(checked["ELO_DIFF"], 0, atol=1e-8)
    ]

    if mismatches.empty:
        print("PASS: All teams have continuous Elo ratings.")
    else:
        print(f"FAIL: {len(mismatches)} continuity mismatches found.")
        display(mismatches)

    return team_games

In [26]:

team_games_2324 = check_elo_continuity(games_2324_elo)
team_games_2425 = check_elo_continuity(games_2425_elo)
team_games_2526 = check_elo_continuity(games_2526_elo)


PASS: All teams have continuous Elo ratings.
PASS: All teams have continuous Elo ratings.
PASS: All teams have continuous Elo ratings.
